# Assignment 2B — RAG Pipeline (HR Policy & Corporate Documents)
**Domain:** HR Policy & Corporate Documents  |  **Generator:** Assignment 1B fine-tuned `HuggingFaceTB/SmolLM2-1.7B-Instruct` (QLoRA Adapter B), with an automatic, evidence-based fall-back to an instruct model  |  **GPU:** Colab T4

**Group 53 — BITS ID, Name, % contribution**

| # | BITS ID | Name | % contribution |
|---|---------|------|----------------|
| 1 | 2025AF05052 | ASHA RANI | 25 % |
| 2 | 2025AF05031 | JOSHI SHRIKANT SHRIDHARRA | 25 % |
| 3 | 2025AE05319 | KOLLI LIKHITHA | 25 % |
| 4 | 2025AF05040 | SHIROLKAR NEEHALI SUNIL | 25 % |

**Pipeline:** cleaned `domain_corpus/*.txt` → token-based chunking → dense search (FAISS) → BM25 + hybrid (RRF) → cross-encoder reranking → grounded generation with chunk citations → LLM-judge (correctness + support) → error analysis.

**What carries over from Assignment 1B (and what is improved):** the cleaned corpus, the 10 evaluation questions with their expected answers (taken from `eval_split.jsonl`) and the fine-tuned model are reused unchanged. The 1B feedback asked for better *response quality, factual validation and corpus organisation*, so here (i) every chunk keeps its source file and character offsets, so each cited chunk can be traced back to a place in the corpus; (ii) every answer is checked against the gold answer span, the expected answer and the retrieved chunks by an independent LLM judge (factual validation); (iii) repetition is *measured* (distinct-2) and removed from the answers; (iv) the generator is switched automatically only if the evidence in D1 shows that the 1B model cannot follow the grounding prompt.

> **Run order:** Runtime → Change runtime type → **T4 GPU** → upload `domain_corpus/`, `eval_split.jsonl`, `train_split.jsonl` (and `hr_adapter/` if you still have it) to `/content` → run the install cell → **Runtime → Restart session** → run everything top to bottom.
> All numbers, tables and inference sentences below are produced by code, so the narrative always matches the measurements.

In [ ]:
# Run once, then restart the runtime (Runtime -> Restart session) and continue from the next cell.
!pip install -q transformers==4.46.3 datasets==2.21.0 huggingface_hub==0.24.7 peft==0.13.2 trl==0.11.4 accelerate==1.0.1 bitsandbytes sentence-transformers==3.3.1 faiss-cpu pandas

In [ ]:
import os, re, glob, json, gc, time, random, math, warnings
from collections import Counter
import numpy as np, pandas as pd, torch
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

assert torch.cuda.is_available(), "GPU required: Runtime -> Change runtime type -> T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
USE_BF16_TRAIN = DTYPE is torch.bfloat16

def sync(): torch.cuda.synchronize()
def free(): gc.collect(); torch.cuda.empty_cache()
def safe_del(*names):
    for n in names: globals().pop(n, None)
    free()

CORPUS_DIR  = "domain_corpus"          # cleaned .txt files from Assignment 1B
EVAL_FILE   = "eval_split.jsonl"       # Assignment 1B evaluation set (question + expected answer)
TRAIN_FILE  = "train_split.jsonl"      # only needed if hr_adapter/ has to be re-trained
ADAPTER_DIR = "hr_adapter"             # Assignment 1B QLoRA adapter (Adapter B)
MODEL_ID    = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

txt_files = sorted(glob.glob(f"{CORPUS_DIR}/*.txt"))
assert len(txt_files) >= 5, f"put the 1B cleaned corpus in ./{CORPUS_DIR}/"
assert os.path.exists(EVAL_FILE), f"put the 1B {EVAL_FILE} in the working directory"
print("GPU:", GPU_NAME, "| dtype:", str(DTYPE).replace("torch.", ""), "| torch", torch.__version__)
print(f"corpus files: {len(txt_files)}"); [print("  -", os.path.basename(f)) for f in txt_files]

## Setup — corpus, embedding model and tokenizers
**Embedding model choice.** The brief's example, `all-MiniLM-L6-v2`, silently truncates every input after **256 tokens**, so a 400-token chunk would lose its second half. The cell below checks this for every chunk size in the study and then uses **`BAAI/bge-small-en-v1.5`** (384-d, **512-token** limit, same cost class) so that *no* chunk size is truncated. This is the "switch to a model with a longer limit" option from the brief. Tokens are counted with the embedding model's own tokenizer.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

EMB_ID = "BAAI/bge-small-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # recommended for bge queries only
emb_model = SentenceTransformer(EMB_ID, device="cuda")
EMB_MAX_TOKENS = emb_model.max_seq_length
emb_tok = emb_model.tokenizer
emb_tok.model_max_length = 10**9                 # silence the 'sequence longer than max' warning; truncation is checked explicitly below
EMB_DIM = emb_model.get_sentence_embedding_dimension()
gen_tok = AutoTokenizer.from_pretrained(MODEL_ID)  # used to count the tokens that are sent to the generator
if gen_tok.pad_token is None: gen_tok.pad_token = gen_tok.eos_token
GEN_CTX = AutoConfig.from_pretrained(MODEL_ID).max_position_embeddings
emb_model.encode(["warm-up"], show_progress_bar=False)   # CUDA warm-up so the first index timing is not inflated

SIZES = [100, 400, 500]       # 100 and 400 are required; 500 is an extra row because the brief's table header says 500
MINILM_LIMIT = 256
print(f"{EMB_ID}: max input {EMB_MAX_TOKENS} tokens, vector size {EMB_DIM}")
print(f"Generator context window: {GEN_CTX} tokens (the 1B adapter was fine-tuned on sequences of at most 512 tokens)")
for s in SIZES:
    print(f"  chunk size {s:>3}: MiniLM (limit {MINILM_LIMIT}) {'would TRUNCATE' if s > MINILM_LIMIT else 'fits'} | {EMB_ID} (limit {EMB_MAX_TOKENS}) {'would TRUNCATE' if s + 2 > EMB_MAX_TOKENS else 'fits'}")

# Corpus: whitespace collapsed once, so chunk text, offsets and answer spans all refer to the same string
def nz(t): return re.sub(r"\s+", " ", t).strip().lower()          # normalisation used for every hit check
CORPUS = {os.path.basename(f): re.sub(r"\s+", " ", open(f, encoding="utf-8").read()).strip() for f in txt_files}
print(f"\ncorpus: {len(CORPUS)} files, {sum(len(t) for t in CORPUS.values()):,} characters, "
      f"{sum(len(emb_tok(t, add_special_tokens=False)['input_ids']) for t in CORPUS.values()):,} embedding-tokens")

## Part A — Dense search and chunk size
### Before you start — the 10 queries and their marked answer spans
The 10 queries and expected answers are taken **verbatim from the Assignment 1B evaluation set** (`eval_split.jsonl`), chosen to cover all source documents and a mix of easy lexical questions and paraphrased ones. For each query the **answer span** is the shortest sentence in the corpus that answers it. A retrieved chunk is a **hit** if it contains that span (compared after lower-casing and whitespace normalisation). The cell asserts that every span really exists in the corpus. An 11th query, whose answer is *not* in the corpus, is used in D1.3.

In [ ]:
QUERIES = [{'id': 'Q1',
  'eval_idx': 20,
  'source': 'HR Mannual- Final (20.09.2018).txt',
  'query': 'How much of the rail fare can an employee encash under the Leave Travel Concession?',
  'expected': 'The encashment of LTC is restricted to 75% of the rail fare as per entitlement.',
  'span': 'The encashment of LTC is restricted to 75% of rail fare as per entitlement.'},
 {'id': 'Q2',
  'eval_idx': 37,
  'source': 'EmployeeHandbook.txt',
  'query': 'When do employees need to submit their savings for Income Tax purposes for the Financial year?',
  'expected': 'Employees are requested to submit their savings for Income Tax purposes for the Financial year on or before 15th November.',
  'span': 'All the employees are requested to submit their savings for Income Tax purposes for the Financial year on or before 15thNovember.'},
 {'id': 'Q3',
  'eval_idx': 148,
  'source': 'EmployeeHandbook.txt',
  'query': 'What is the minimum balance that should remain after encashing earned leave?',
  'expected': 'The balance at credit should not be less than 30 days after deducting the encashed earned leave.',
  'span': 'The balance at credit should not be less than 30 days after deducting the encashed EL.'},
 {'id': 'Q4',
  'eval_idx': 76,
  'source': 'd0d7d238-ab60-4a0e-bd1a-c1a1e6d45b89_HR-policy-IHLL.txt',
  'query': 'Is there a way to get paid for unused annual leave?',
  'expected': 'No, the Company will not provide compensation in lieu of unused annual leave.',
  'span': 'The Company will not provide compensation in lieu of unused annual leave.'},
 {'id': 'Q5',
  'eval_idx': 81,
  'source': 'd0d7d238-ab60-4a0e-bd1a-c1a1e6d45b89_HR-policy-IHLL.txt',
  'query': 'What action must be taken for network services that are not required on the servers?',
  'expected': 'All network services that are not required on the servers must be disabled.',
  'span': 'All network services that are not required on the servers must be disabled.'},
 {'id': 'Q6',
  'eval_idx': 118,
  'source': 'hr_manual.txt',
  'query': 'What is the standard practice for booking flights under this travel policy?',
  'expected': 'The cheapest flight within the time range shall be booked as a standard practice.',
  'span': 'The cheapest flight within the time range shall be booked as a standard practice.'},
 {'id': 'Q7',
  'eval_idx': 35,
  'source': 'HR-POLICY-EMPLOYEE-HANDBOOK.txt',
  'query': 'What is the responsibility of a faculty member during invigilation duty?',
  'expected': 'A faculty member must follow the instructions relating to the invigilation duty assigned to them by the Examination Section.',
  'span': 'A faculty must follow the instructions relating to the invigilation duty assigned to him / her by the Examination Section.'},
 {'id': 'Q8',
  'eval_idx': 25,
  'source': 'hr_manual.txt',
  'query': 'What does the HR manual state about the leave year?',
  'expected': 'The leave year is as per the calendar year (1st January – 31st December).',
  'span': 'Leave year is as per the calendar year (1st January – 31st December).'},
 {'id': 'Q9',
  'eval_idx': 150,
  'source': 'HR AFR - Leave Policy.txt',
  'query': "What is the company's stance on working on public holidays?",
  'expected': 'Employees shall not be required to work on public holidays and do not need to apply for leave to justify their absence.',
  'span': 'Employees shall not be required to work on public holidays and shall not need to apply for leave to justify such absence from work on public holidays.'},
 {'id': 'Q10',
  'eval_idx': 104,
  'source': 'HR MEI – India Higher Education Instructions.txt',
  'query': 'What happens to the registration and admission fees if an employee successfully completes the program?',
  'expected': 'The company will refund the one-time registration and admission fee upon successful program completion.',
  'span': 'Company will refund one time registration and admission fee upon successful program completion.'}]

OOS = dict(id="Q11", eval_idx=None, source="(not in corpus)",
           query="What is the company's policy on paying employees' salaries in cryptocurrency?",
           expected="Not found in the documents", span=None)

# 1) the queries and expected answers must be exactly those of the 1B evaluation set
eval_rows = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8")]
eval_map = {r["instruction"]: r["response"] for r in eval_rows}
for q in QUERIES:
    assert q["query"] in eval_map and eval_map[q["query"]] == q["expected"], f"{q['id']} is not in {EVAL_FILE} - wrong eval file?"

# 2) every span must be found in the corpus
rows = []
for q in QUERIES:
    found = [f for f, t in CORPUS.items() if nz(q["span"]) in nz(t)]
    assert found, f"{q['id']}: span not found in corpus"
    q["span_files"] = found
    rows.append({"Query": q["id"], "Question": q["query"], "Source file(s) containing the span": ", ".join(found),
                 "Span tokens": len(emb_tok(q["span"], add_special_tokens=False)["input_ids"]),
                 "Answer span": q["span"]})
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 250)
span_df = pd.DataFrame(rows); display(span_df)
print("longest span:", span_df["Span tokens"].max(), "tokens (a 100-token chunk can only be a hit if the whole span falls inside one chunk)")

# 3) the out-of-corpus query really has no answer in the corpus
OOS_KEYWORDS = ["cryptocurrenc", "crypto", "bitcoin"]
assert not any(k in nz(t) for t in CORPUS.values() for k in OOS_KEYWORDS), "OOS query is answerable - choose another"
print("OOS query check passed: none of", OOS_KEYWORDS, "occurs in the corpus")
span_df.to_csv("Part_A_answer_spans.csv", index=False)

### A1 — Dense search
`build_dense_search(chunks, model, ...)` is the one reusable function: it embeds a set of chunks in batches, L2-normalises the vectors, builds a FAISS `IndexFlatIP` (inner product on unit vectors = **cosine similarity**) and returns an object whose `.search(query, k)` gives the top-k chunk ids and scores (`.top_chunks(query, k)` returns the chunk records themselves). Build time is measured around embedding + indexing.

In [ ]:
import faiss
BATCH_SIZE = 64

class DenseSearcher:
    def __init__(self, chunks, model, batch_size=BATCH_SIZE, query_prefix=""):
        self.chunks, self.model, self.prefix, self.batch_size = chunks, model, query_prefix, batch_size
        sync(); t0 = time.perf_counter()
        emb = model.encode([c["text"] for c in chunks], batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False).astype("float32")
        self.index = faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb)
        sync(); self.build_seconds = time.perf_counter() - t0
        self.dim = emb.shape[1]
    def search(self, query, k=5):
        qv = self.model.encode([self.prefix + query], normalize_embeddings=True, convert_to_numpy=True,
                               show_progress_bar=False).astype("float32")
        scores, ids = self.index.search(qv, k)
        keep = [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]
        return [i for i, _ in keep], [s for _, s in keep]
    def top_chunks(self, query, k=5):
        return [self.chunks[i] for i in self.search(query, k)[0]]

def build_dense_search(chunks, model, batch_size=BATCH_SIZE, query_prefix=""):
    """Embed chunks -> FAISS IndexFlatIP -> object with .search(query, k)."""
    return DenseSearcher(chunks, model, batch_size, query_prefix)

def chunk_fixed(size, min_tokens=10):
    """Fixed-size chunks of at most `size` embedding-model tokens, no overlap, one document at a time (never across files).
    Each cut is moved back to the nearest word start (so no word is split in two chunks); chunks are therefore `size` tokens
    or a few tokens fewer, never more. Chunk text is cut from the original string with the tokenizer's character offsets.
    A trailing remainder shorter than `min_tokens` tokens is dropped (pure noise)."""
    chunks = []
    for fname, text in CORPUS.items():
        offs = emb_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        n = len(offs)
        word_start = lambda j: j <= 0 or j >= n or text[offs[j][0] - 1].isspace()
        start = 0
        while start < n:
            end = min(start + size, n)
            if end < n:
                e = end
                while e > start + size // 2 and not word_start(e): e -= 1
                if word_start(e): end = e
            if end - start >= min_tokens:
                a, b = offs[start][0], offs[end - 1][1]
                chunks.append(dict(id=len(chunks), source=fname, char_start=a, char_end=b, n_tokens=end - start, text=text[a:b]))
            start = end
    return chunks

# helpers shared by Parts A-D
SPANS = {q["id"]: nz(q["span"]) for q in QUERIES}
QI = {q["id"]: q for q in QUERIES}
def is_hit(chunk_text, qid): return SPANS[qid] in nz(chunk_text)
_gen_tok_cache = {}
def gen_tokens(text):
    if text not in _gen_tok_cache: _gen_tok_cache[text] = len(gen_tok(text, add_special_tokens=False)["input_ids"])
    return _gen_tok_cache[text]
print("functions defined")

### A2 — Choose a chunk size
Fixed-size chunking at **100** and **400** tokens (no overlap; each cut is moved back to the nearest word start so no word is split), plus **500** as an extra row. For each size the same dense search is run on the 10 queries. *Hit rate@3* = share of queries where one of the top-3 chunks contains the answer span; *tokens sent per query* = combined size of the top-3 chunks (counted with the **generator's** tokenizer, because that is what the generator has to read), averaged over the queries. The "chunks over model limit" column proves that no chunk is truncated by the embedding model.

In [ ]:
size_runs, rows = {}, []
for size in SIZES:
    ch = chunk_fixed(size)
    over = sum(len(emb_tok(c["text"])["input_ids"]) > EMB_MAX_TOKENS for c in ch)       # incl. [CLS]/[SEP]
    searcher = build_dense_search(ch, emb_model, BATCH_SIZE, BGE_QUERY_PREFIX)
    hits, toks = 0, []
    for q in QUERIES:
        ids = searcher.search(q["query"], 3)[0]
        hits += any(is_hit(ch[i]["text"], q["id"]) for i in ids)
        toks.append(sum(gen_tokens(ch[i]["text"]) for i in ids))
    size_runs[size] = dict(chunks=ch, searcher=searcher, over=over, hit3=100 * hits / len(QUERIES), toks=float(np.mean(toks)))
    rows.append({"Chunk size (tokens)": size, "Total chunks": len(ch), "Hit rate@3 (%)": size_runs[size]["hit3"],
                 "Tokens sent per query (top 3)": round(size_runs[size]["toks"], 1),
                 "Chunks over model limit": over, "Index build time (s)": round(searcher.build_seconds, 2)})
a2_df = pd.DataFrame(rows); display(a2_df); a2_df.to_csv("Part_A2_chunk_size_table.csv", index=False)
assert (a2_df[a2_df["Chunk size (tokens)"] <= 400]["Chunks over model limit"] == 0).all(), "a chunk exceeds the embedding model's limit"
if (a2_df["Chunks over model limit"] > 0).any(): print("note: some chunks exceed the model limit by a few tokens (only the last few tokens are cut off)")

**Choice rule (applied by code):** the eligible sizes are the two required ones, 100 and 400 (the extra 500-token row is shown because the brief's table header lists it, but 500 tokens + the question would exceed the 512-token input of the cross-encoder used in Part C). With only 10 queries, one query = 10 percentage points, so every size whose hit rate@3 is within **one query** of the best is treated as equally accurate; among those, the size that sends the **fewest tokens** wins (less context for the 1.7B generator to read). The sentences below are generated from the table.

In [ ]:
ELIGIBLE = [100, 400]      # the 500-token row is shown for the brief's table header only: 500 tokens + the question would not fit the 512-token reranker input
elig = a2_df[a2_df["Chunk size (tokens)"].isin(ELIGIBLE)]
best_h = elig["Hit rate@3 (%)"].max()
cand = elig[elig["Hit rate@3 (%)"] >= best_h - 100 / len(QUERIES)]
CHUNK_SIZE = int(cand.sort_values("Tokens sent per query (top 3)").iloc[0]["Chunk size (tokens)"])
CHUNKS, DENSE = size_runs[CHUNK_SIZE]["chunks"], size_runs[CHUNK_SIZE]["searcher"]
chosen_h = size_runs[CHUNK_SIZE]["hit3"]
others = elig[elig["Chunk size (tokens)"] != CHUNK_SIZE]
cmp_txt = "; ".join(f"{int(r['Chunk size (tokens)'])} tokens reaches {r['Hit rate@3 (%)']:.0f}% while sending {r['Tokens sent per query (top 3)']:.0f} tokens"
                    for _, r in others.iterrows())
prompt_overhead = 160          # instructions + question + labels in the D1 prompt (the real length is printed in D1)
need = size_runs[CHUNK_SIZE]["toks"] + prompt_overhead
fit = "inside" if need <= 512 else "beyond"
CHUNK_CHOICE_TEXT = (f"I use **{CHUNK_SIZE}-token chunks** in Parts B to D: they reach a hit rate@3 of {chosen_h:.0f}% (best of the eligible sizes: {best_h:.0f}%; a gap of one query is within noise on 10 queries) "
                     f"while sending only {size_runs[CHUNK_SIZE]['toks']:.0f} tokens per query, whereas {cmp_txt}. "
                     f"With the ~{prompt_overhead}-token prompt (instructions, question, labels), the generator reads about {need:.0f} tokens per question, which is {fit} the 512-token sequences the "
                     f"Assignment 1 adapter was fine-tuned on (the model itself accepts {GEN_CTX}), so "
                     + ("the context stays close to what the model saw in training." if fit == "inside" else "the 1B model's grounding quality has to be checked in D1 rather than assumed."))
print("CHOSEN CHUNK SIZE:", CHUNK_SIZE, "tokens ->", len(CHUNKS), "chunks")
from IPython.display import Markdown
display(Markdown("**A2 (2) — Chunk size decision:** " + CHUNK_CHOICE_TEXT))

### A1 (2) — Report for the final chunk set
Placed here because it describes the index of the *chosen* chunk size.

In [ ]:
a1_df = pd.DataFrame([
    ["Embedding model", EMB_ID],
    ["Vector size", f"{DENSE.dim}"],
    ["Similarity measure", "cosine similarity (L2-normalised vectors + FAISS IndexFlatIP inner product)"],
    ["Batch size (encoding)", f"{DENSE.batch_size}"],
    ["Embedding model max input", f"{EMB_MAX_TOKENS} tokens ({size_runs[CHUNK_SIZE]['over']} chunks of the final set exceed it)"],
    ["Final chunk set", f"{CHUNK_SIZE}-token fixed chunks, no overlap, {len(CHUNKS)} chunks"],
    ["Index build time (embed + index)", f"{DENSE.build_seconds:.2f} s on {GPU_NAME}"],
], columns=["Item", "Value"])
display(a1_df); a1_df.to_csv("Part_A1_index_report.csv", index=False)

## Part B — Keyword and hybrid search
Same chunks, same 10 queries, top-5 each time. **Dense** = Part A search. **BM25** = Okapi BM25 (k1 = 1.5, b = 0.75, lower-cased alphanumeric tokens, small stop-word list, plural-"s" folding), written out in ~20 lines so there is no extra dependency. **Hybrid** = Reciprocal Rank Fusion of the **top 20** of each list: `RRF(d) = 1/(60 + rank_dense) + 1/(60 + rank_bm25)`, ranks start at 1, a chunk missing from a list gets nothing from it; the top 5 are kept. Latency = time to return the top 5 for one query (dense includes embedding the query; hybrid includes both searches and the fusion), averaged over the 10 queries after one warm-up run.

In [ ]:
STOP = set("a an and are as at be been by for from has have if in into is it its of on or that the their there these they this to was were will with you your we our".split())
def bm25_tokens(text):
    out = []
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if w in STOP: continue
        if len(w) > 3 and w.endswith("s") and not w.endswith("ss"): w = w[:-1]      # holidays -> holiday
        out.append(w)
    return out

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.N = len(docs_tokens); self.k1, self.b = k1, b
        self.dl = np.array([len(d) for d in docs_tokens], dtype=float); self.avgdl = self.dl.mean()
        self.post = {}
        for i, d in enumerate(docs_tokens):
            for t, c in Counter(d).items(): self.post.setdefault(t, []).append((i, c))
        self.idf = {t: math.log(1 + (self.N - len(p) + 0.5) / (len(p) + 0.5)) for t, p in self.post.items()}
    def scores(self, q_tokens):
        s = np.zeros(self.N)
        for t in set(q_tokens):
            p = self.post.get(t)
            if not p: continue
            idx = np.array([i for i, _ in p]); tf = np.array([c for _, c in p], dtype=float)
            s[idx] += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.dl[idx] / self.avgdl))
        return s
    def top(self, q_tokens, k):
        s = self.scores(q_tokens); order = np.argsort(-s, kind="stable")[:k]
        return [int(i) for i in order if s[i] > 0], [float(s[i]) for i in order if s[i] > 0]

sync(); t0 = time.perf_counter()
BM25_IDX = BM25([bm25_tokens(c["text"]) for c in CHUNKS])
print(f"BM25 index: {BM25_IDX.N} chunks, {len(BM25_IDX.post)} distinct terms, built in {1000*(time.perf_counter()-t0):.0f} ms")

def rrf_fuse(dense_ids, bm25_ids, top=5, k=60):
    score = {}
    for r, i in enumerate(dense_ids, 1): score[i] = score.get(i, 0.0) + 1.0 / (k + r)
    for r, i in enumerate(bm25_ids, 1):  score[i] = score.get(i, 0.0) + 1.0 / (k + r)
    drank = {i: r for r, i in enumerate(dense_ids, 1)}
    return sorted(score, key=lambda i: (-score[i], drank.get(i, 10**9), i))[:top]

def search_dense(q, k=5):  return DENSE.search(q, k)[0]
def search_bm25(q, k=5):   return BM25_IDX.top(bm25_tokens(q), k)[0]
def search_hybrid(q, k=5, pool=20):
    return rrf_fuse(DENSE.search(q, pool)[0], BM25_IDX.top(bm25_tokens(q), pool)[0], top=k)
METHODS = {"Dense": search_dense, "BM25": search_bm25, "Hybrid (RRF)": search_hybrid}

# sanity check of the fusion formula on a toy example: chunk 7 is 2nd in both lists, chunk 1 is 1st in only one
toy = rrf_fuse([1, 7, 3], [9, 7, 4], top=3)
assert toy[0] == 7 and set(toy[1:]) == {1, 9}, "RRF check failed"       # 7: 2/62 > 1/61 (rank 1 in one list only)
print("RRF toy check passed ->", toy)

In [ ]:
def timed_run(fn, k=5):
    fn(QUERIES[0]["query"], k)                       # one warm-up run, not timed
    lat, res = [], []
    for q in QUERIES:
        sync(); t0 = time.perf_counter(); ids = fn(q["query"], k); sync()
        lat.append(1000 * (time.perf_counter() - t0)); res.append(ids)
    return res, float(np.mean(lat))

def hit_rate(res, k):
    return 100 * np.mean([any(is_hit(CHUNKS[i]["text"], q["id"]) for i in ids[:k]) for q, ids in zip(QUERIES, res)])

B_RES, B_LAT = {}, {}
rows = []
for name, fn in METHODS.items():
    B_RES[name], B_LAT[name] = timed_run(fn, 5)
    rows.append({"Method": name, "Average query latency (ms)": round(B_LAT[name], 2),
                 "Hit rate@1 (%)": hit_rate(B_RES[name], 1), "Hit rate@3 (%)": hit_rate(B_RES[name], 3),
                 "Hit rate@5 (%) [extra]": hit_rate(B_RES[name], 5)})
b_df = pd.DataFrame(rows); display(b_df); b_df.to_csv("Part_B_search_comparison.csv", index=False)

**Where does each method place the answer chunk?** First rank (within the top 20) of a chunk that contains the answer span; "–" = not in the top 20. This per-query view is what the comparison sentence below is based on.

In [ ]:
def first_hit_rank(fn, q, depth=20):
    for r, i in enumerate(fn(q["query"], depth), 1):
        if is_hit(CHUNKS[i]["text"], q["id"]): return r
    return None
RANKS = {name: [first_hit_rank(fn, q) for q in QUERIES] for name, fn in METHODS.items()}
rk_df = pd.DataFrame({"Query": [q["id"] for q in QUERIES], "Question": [q["query"][:70] for q in QUERIES],
                      **{n: [r if r else "–" for r in v] for n, v in RANKS.items()}})
display(rk_df); rk_df.to_csv("Part_B_first_hit_ranks.csv", index=False)

BEST_METHOD = max(METHODS, key=lambda m: (b_df.set_index("Method").loc[m, "Hit rate@1 (%)"],
                                         b_df.set_index("Method").loc[m, "Hit rate@3 (%)"], -B_LAT[m]))
INF = 10**6
def rk(m, i): return RANKS[m][i] or INF
gap = []
for i, q in enumerate(QUERIES):
    others_ = [rk(m, i) for m in METHODS if m != BEST_METHOD]
    gap.append((min(max(others_), INF) - rk(BEST_METHOD, i), i) if rk(BEST_METHOD, i) < INF else (-INF, i))
ex_i = max(gap)[1]; ex_q = QUERIES[ex_i]
fmt = lambda m: (f"rank {RANKS[m][ex_i]}" if RANKS[m][ex_i] else "not in its top 20")
WHY_B = {"Dense": "it matches paraphrased questions to the policy wording by meaning",
         "BM25": "the policy questions share rare exact words (numbers, 'LTC', 'EL', 'invigilation') with the answer sentence, and BM25 weights exactly those",
         "Hybrid (RRF)": "a chunk that is ranked fairly well by both the meaning-based and the keyword-based list rises above chunks that only one of them likes"}
bt = b_df.set_index("Method")
BEST_METHOD_TEXT = (f"**{BEST_METHOD}** works best on these 10 queries (hit rate@1 {bt.loc[BEST_METHOD,'Hit rate@1 (%)']:.0f}%, @3 {bt.loc[BEST_METHOD,'Hit rate@3 (%)']:.0f}%, "
                    f"{B_LAT[BEST_METHOD]:.1f} ms per query), because {WHY_B[BEST_METHOD]}. "
                    f"Example, {ex_q['id']} (\"{ex_q['query']}\"): " + ", ".join(f"{m} gives {fmt(m)}" for m in METHODS) + ". "
                    f"With only 10 queries one query is worth 10 percentage points, so small differences between methods are not conclusive.")
display(Markdown("**B4 — Best method:** " + BEST_METHOD_TEXT))
print("BEST_METHOD used in Part C:", BEST_METHOD)

## Part C — Reranking
The best method from Part B retrieves the **top 10**; a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) reads each *(query, chunk)* pair jointly and re-scores it; the top 3 are kept. Reranking time covers only the cross-encoder scoring + sorting of the 10 candidates (one warm-up run first).

In [ ]:
from sentence_transformers import CrossEncoder
CE_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ce = CrossEncoder(CE_ID, max_length=512, device="cuda")
BEST_FN = METHODS[BEST_METHOD]

def rerank(query, ids):
    sc = ce.predict([(query, CHUNKS[i]["text"]) for i in ids], batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(sc), kind="stable")
    return [ids[j] for j in order], [float(sc[j]) for j in order]

def pipeline_top3(query):
    """retrieve top-10 with the best method -> cross-encoder -> top 3 (used for every query in Part D)."""
    cand = BEST_FN(query, 10)
    return rerank(query, cand)[0][:3]

rerank(QUERIES[0]["query"], BEST_FN(QUERIES[0]["query"], 10))        # warm-up
BEFORE, AFTER, RR_MS = {}, {}, []
for q in QUERIES:
    cand = BEST_FN(q["query"], 10)
    sync(); t0 = time.perf_counter(); new_order, _ = rerank(q["query"], cand); sync()
    RR_MS.append(1000 * (time.perf_counter() - t0))
    BEFORE[q["id"]], AFTER[q["id"]] = cand[:3], new_order[:3]
print(f"cross-encoder: {CE_ID} | retrieval method: {BEST_METHOD} (top 10 -> top 3)")

### C2 — Top 3 before and after reranking (✓ = chunk contains the answer span)

In [ ]:
def snippet(i, qid, width=110):
    t = CHUNKS[i]["text"]; pos = nz(t).find(SPANS[qid])
    return ("…" + nz(t)[max(0, pos - 25):pos + width] + "…") if pos >= 0 else t[:width] + "…"      # hit: show the text around the answer span
def line(i, qid, rank):
    c = CHUNKS[i]
    return f"   #{rank} chunk {i:<5} {'✓' if is_hit(c['text'], qid) else '✗'}  [{c['source'][:22]}]  {snippet(i, qid)}"
for q in QUERIES:
    print(f"{q['id']}: {q['query']}")
    print("  BEFORE reranking:"); [print(line(i, q['id'], r)) for r, i in enumerate(BEFORE[q['id']], 1)]
    print("  AFTER reranking: "); [print(line(i, q['id'], r)) for r, i in enumerate(AFTER[q['id']], 1)]
    print("  first result changed:", BEFORE[q['id']][0] != AFTER[q['id']][0], "\n")

### C3 — Reranking measures

In [ ]:
def h(d, k): return 100 * np.mean([any(is_hit(CHUNKS[i]["text"], q["id"]) for i in d[q["id"]][:k]) for q in QUERIES])
CHANGE = 100 * np.mean([BEFORE[q["id"]][0] != AFTER[q["id"]][0] for q in QUERIES])
H1_B_FROM_PART_B = float(b_df.set_index("Method").loc[BEST_METHOD, "Hit rate@1 (%)"])
assert abs(H1_B_FROM_PART_B - h(BEFORE, 1)) < 1e-9, "hit@1 before reranking differs from Part B"
H1_A, H3_B, H3_A = h(AFTER, 1), h(BEFORE, 3), h(AFTER, 3)
c_df = pd.DataFrame([
    ["Reranking time per query (ms)", round(float(np.mean(RR_MS)), 2)],
    ["Rank change rate (%)", CHANGE],
    ["Hit rate@1 before reranking (from Part B)", H1_B_FROM_PART_B],
    ["Hit rate@1 after reranking", H1_A],
    ["Hit rate@3 before reranking [extra]", H3_B],
    ["Hit rate@3 after reranking [extra]", H3_A],
    [f"Retrieval time, {BEST_METHOD}, top 10 (ms) [extra]", round(B_LAT[BEST_METHOD], 2)],
], columns=["Measure", "Value"])
display(c_df); c_df.to_csv("Part_C_reranking_table.csv", index=False)

verdict = "helped" if H1_A > H1_B_FROM_PART_B else ("did not change accuracy for" if H1_A == H1_B_FROM_PART_B else "hurt")
RERANK_TEXT = (f"Reranking {verdict} this domain: hit rate@1 went from {H1_B_FROM_PART_B:.0f}% to {H1_A:.0f}% (hit rate@3 {H3_B:.0f}% → {H3_A:.0f}%) and the first result changed for {CHANGE:.0f}% of the queries, "
               f"at a cost of {np.mean(RR_MS):.1f} ms per query on top of {B_LAT[BEST_METHOD]:.1f} ms retrieval. "
               + ("The extra accuracy is worth the added milliseconds because the generator only sees the top 3 chunks." if H1_A > H1_B_FROM_PART_B
                  else "Since accuracy did not improve on these 10 queries, the extra latency is only justified if it protects against harder questions than this set contains." if H1_A == H1_B_FROM_PART_B
                  else "The cross-encoder pushed the answer chunk down on at least one query, so on this corpus the added time is not justified without a domain-tuned reranker."))
display(Markdown("**C4 — Did reranking help?** " + RERANK_TEXT))

## Part D — Generate and check answers
### D1 — Grounded answers
**D1.1 The prompt.** One fixed prompt template for every query: answer only from the numbered chunks, cite the chunk numbers used (e.g. `[2]`), and reply exactly `Not found in the documents` if the chunks do not contain the answer. It is sent as a plain user turn in the same ChatML format (no system prompt) the Assignment 1 adapter was trained with.

In [ ]:
RAG_TEMPLATE = """You are an HR policy assistant. Answer the question using ONLY the numbered chunks below.
Rules:
1. Use only facts that are written in the chunks. Do not use outside knowledge.
2. After every statement, cite the chunk number(s) you used in square brackets, for example [2].
3. If the chunks do not contain the answer, reply exactly: Not found in the documents
4. Answer in 1 to 3 sentences and do not repeat yourself.

Chunks:
<<CHUNKS>>

Question: <<QUESTION>>
Answer:"""
def build_rag_message(question, chunk_texts):
    block = "\n\n".join(f"[{n}] {t}" for n, t in enumerate(chunk_texts, 1))
    return RAG_TEMPLATE.replace("<<CHUNKS>>", block).replace("<<QUESTION>>", question)

print("=== D1.1  GENERATION PROMPT (template) ===\n"); print(RAG_TEMPLATE)

TOP3 = {q["id"]: AFTER[q["id"]] for q in QUERIES}          # top-3 AFTER reranking, from Part C
TOP3[OOS["id"]] = pipeline_top3(OOS["query"])              # same pipeline for the out-of-corpus query
PROMPT_TOKENS = [gen_tokens(build_rag_message(q["query"], [CHUNKS[i]["text"] for i in TOP3[q["id"]]])) for q in QUERIES]
print(f"\nfull prompt length over the 10 queries: mean {np.mean(PROMPT_TOKENS):.0f} tokens, max {max(PROMPT_TOKENS)} (adapter fine-tuned on <= 512; model window {GEN_CTX})")

**Generator 1 = the Assignment 1 model** (SmolLM2-1.7B-Instruct + QLoRA Adapter B). If `hr_adapter/` is not present it is re-trained here with exactly the 1B recipe (r = 16, α = 32, `q_proj`/`v_proj`, 3 epochs, lr 2e-4, response-only loss) from `train_split.jsonl` — a few minutes on a T4.

In [ ]:
def build_prompt(user_msg):   # ChatML WITHOUT a system prompt, exactly as in 1B
    return f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
def build_sft_text(instr, resp): return build_prompt(instr) + f"{resp}<|im_end|>\n"

if not os.path.isdir(ADAPTER_DIR):
    assert os.path.exists(TRAIN_FILE), f"neither {ADAPTER_DIR}/ nor {TRAIN_FILE} found"
    print("hr_adapter/ not found -> re-training Adapter B exactly as in Assignment 1B")
    from datasets import Dataset
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
    tokenizer = gen_tok
    ld = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
    tr_ds = Dataset.from_list([{"text": build_sft_text(r["instruction"], r["response"])} for r in ld(TRAIN_FILE)])
    ev_ds = Dataset.from_list([{"text": build_sft_text(r["instruction"], r["response"])} for r in ld(EVAL_FILE)])
    bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
    qmodel = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb4, torch_dtype=DTYPE, device_map={"": 0})
    qmodel.config.use_cache = False
    qmodel = prepare_model_for_kbit_training(qmodel, use_gradient_checkpointing=True)
    lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    resp_ids = gen_tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
    collator = DataCollatorForCompletionOnlyLM(response_template=resp_ids, tokenizer=gen_tok)
    sft_args = SFTConfig(output_dir="hr_qlora_model", per_device_train_batch_size=8, per_device_eval_batch_size=8,
        gradient_accumulation_steps=2, learning_rate=2e-4, num_train_epochs=3, lr_scheduler_type="cosine", warmup_ratio=0.05,
        logging_steps=5, eval_strategy="epoch", save_strategy="no", bf16=USE_BF16_TRAIN, fp16=not USE_BF16_TRAIN,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}, max_seq_length=512,
        dataset_text_field="text", report_to="none", seed=SEED)
    trainer = SFTTrainer(model=qmodel, args=sft_args, train_dataset=tr_ds, eval_dataset=ev_ds, tokenizer=gen_tok,
                         peft_config=lora_cfg, data_collator=collator)
    t0 = time.time(); trainer.train(); print(f"training time {(time.time()-t0)/60:.1f} min")
    trainer.model.save_pretrained(ADAPTER_DIR); gen_tok.save_pretrained(ADAPTER_DIR)
    safe_del("trainer", "qmodel")
else:
    print("using existing", ADAPTER_DIR)

In [ ]:
from peft import PeftModel
@torch.no_grad()
def chat(model, tok, user_msg, style, max_new_tokens=150, eos_id=None):
    """Greedy decoding. style='chatml' = 1B format without system prompt; style='template' = tokenizer chat template."""
    text = build_prompt(user_msg) if style == "chatml" else tok.apply_chat_template([{"role": "user", "content": user_msg}], tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt", add_special_tokens=(style == "chatml")).to(model.device)   # chat templates already contain BOS
    kw = dict(eos_token_id=eos_id) if eos_id is not None else {}
    sync(); t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id, **kw)
    sync(); dt = time.perf_counter() - t0
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip(), dt

NF_RE = re.compile(r"not found in the documents", re.I)
def words(t): return re.findall(r"[a-z0-9']+", t.lower())
def distinct_2(t):
    w = words(t); bi = list(zip(w, w[1:])); return round(len(set(bi)) / max(1, len(bi)), 3)
def dedup_sentences(text):
    """Remove sentences repeated verbatim (ignoring [n] citations) - the repetition loop seen in 1B."""
    seen, out = set(), []
    for p in re.split(r"(?<=[.!?])\s+", text.strip()):
        key = re.sub(r"\W+", " ", re.sub(r"\[\d+\]", "", p.lower())).strip()
        if key and key in seen: continue
        seen.add(key); out.append(p)
    return " ".join(out)

def run_generator(model, tok, style, eos_id):
    rows = []
    for q in QUERIES + [OOS]:
        ids = TOP3[q["id"]]
        raw, dt = chat(model, tok, build_rag_message(q["query"], [CHUNKS[i]["text"] for i in ids]), style, 150, eos_id)
        ans = dedup_sentences(raw)
        cites = sorted({int(x) for x in re.findall(r"\[(\d+)\]", ans)})
        rows.append(dict(id=q["id"], query=q["query"], chunk_ids=ids, raw=raw, answer=ans, cites=[c for c in cites if 1 <= c <= 3],
                         bad_cites=[c for c in cites if not 1 <= c <= 3], not_found=bool(NF_RE.search(ans)),
                         distinct_2_raw=distinct_2(raw), distinct_2=distinct_2(ans), seconds=round(dt, 2)))
    return rows

def show_answers(rows, title):
    print("=" * 100); print(title); print("=" * 100)
    for r in rows:
        q = QI.get(r["id"], OOS)
        hit = [("✓" if (r["id"] in SPANS and is_hit(CHUNKS[i]["text"], r["id"])) else "✗") for i in r["chunk_ids"]]
        print(f"\n{r['id']}: {r['query']}\n   chunks given: {[f'[{n}] id {i} {h}' for n, (i, h) in enumerate(zip(r['chunk_ids'], hit), 1)]}   (✓ = contains the answer span)")
        print(f"   ANSWER: {r['answer']}")
        print(f"   cites={r['cites']} not_found={r['not_found']} distinct-2={r['distinct_2']} (raw {r['distinct_2_raw']})" + ("  | repetition removed" if r['raw'] != r['answer'] else ""))
    print()

ft_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0})
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_DIR).eval()
IM_END = gen_tok.convert_tokens_to_ids("<|im_end|>")
chat(ft_model, gen_tok, "Hello", "chatml", 5, IM_END)     # warm-up
FT_ROWS = run_generator(ft_model, gen_tok, "chatml", IM_END)
json.dump(FT_ROWS, open("D1_answers_assignment1_model.json", "w"), indent=1, ensure_ascii=False)
safe_del("ft_model", "ft_base")

**D1.2 / D1.3 — Assignment 1 model: all 10 answers + the "Not found" query.**

In [ ]:
show_answers(FT_ROWS, "Assignment 1 model (SmolLM2-1.7B-Instruct + QLoRA Adapter B) - top-3 reranked chunks")

**Does the Assignment 1 model follow the grounding prompt?** The brief allows switching to an instruct model if the 1B model ignores the chunks or never says "Not found". The decision is made by code from two measured conditions: (a) it must reply "Not found in the documents" on the out-of-corpus query; (b) it must cite chunk numbers in at least half of the 10 in-corpus answers (an answer without any citation is not grounded in labelled chunks). `FORCE_GENERATOR` can override the decision.

In [ ]:
FORCE_GENERATOR = None        # None = decide automatically | "ft" = keep the 1B model | "qwen" = force Qwen2.5-7B-Instruct
oos_row = FT_ROWS[-1]
cite_rate = np.mean([len(r["cites"]) > 0 for r in FT_ROWS[:10]])
cond_a, cond_b = oos_row["not_found"], cite_rate >= 0.5
SWITCH = (not (cond_a and cond_b)) if FORCE_GENERATOR is None else (FORCE_GENERATOR == "qwen")
print(f"(a) says 'Not found' on the out-of-corpus query: {cond_a} | (b) cites chunk numbers in {100*cite_rate:.0f}% of the 10 answers (needs >= 50%)")
if not cond_a:
    print("\nEXAMPLE OF THE PROBLEM - out-of-corpus query, the 1B model did not say 'Not found in the documents':")
    print("  Q:", oos_row["query"]); print("  A:", oos_row["answer"])
elif not cond_b:
    r = next(r for r in FT_ROWS[:10] if not r["cites"])
    print("\nEXAMPLE OF THE PROBLEM - answer without any chunk citation:"); print("  Q:", r["query"]); print("  A:", r["answer"])
print("\nDECISION:", "switch to Qwen/Qwen2.5-7B-Instruct (4-bit)" if SWITCH else "keep the Assignment 1 model as generator")

In [ ]:
GEN_ID = "Qwen/Qwen2.5-7B-Instruct"
if SWITCH:
    q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
    qtok = AutoTokenizer.from_pretrained(GEN_ID)
    qmodel_g = AutoModelForCausalLM.from_pretrained(GEN_ID, quantization_config=q4, torch_dtype=DTYPE, device_map={"": 0}).eval()
    qeos = qtok.convert_tokens_to_ids("<|im_end|>")
    chat(qmodel_g, qtok, "Hello", "template", 5, qeos)
    FINAL_ROWS = run_generator(qmodel_g, qtok, "template", qeos); FINAL_NAME = GEN_ID + " (4-bit NF4)"
    safe_del("qmodel_g")
else:
    FINAL_ROWS, FINAL_NAME = FT_ROWS, "SmolLM2-1.7B-Instruct + QLoRA Adapter B (Assignment 1B)"
json.dump(FINAL_ROWS, open("D1_answers_final_generator.json", "w"), indent=1, ensure_ascii=False)
FINAL_IS_QWEN = SWITCH
print("FINAL GENERATOR:", FINAL_NAME)
show_answers(FINAL_ROWS, f"FINAL GENERATOR: {FINAL_NAME} - all 10 answers + the out-of-corpus query (Q11)")
d1 = pd.DataFrame([{"Query": r["id"], "Answer": r["answer"], "Cited chunks": r["cites"], "Says 'Not found'": r["not_found"], "distinct-2": r["distinct_2"]} for r in FINAL_ROWS])
d1.to_csv("D1_answers_table.csv", index=False)
n_nf = FINAL_ROWS[-1]["not_found"]
display(Markdown(f"**D1.3 — Not-found query:** with {FINAL_NAME}, the out-of-corpus question ({OOS['id']}) " + ("**was answered with 'Not found in the documents'**, as required." if n_nf else "**was NOT answered with 'Not found in the documents'** - see the answer above.")
                 + f" Repetition (distinct-2, higher = less repetitive): mean {np.mean([r['distinct_2'] for r in FINAL_ROWS[:10]]):.3f} over the 10 final answers (raw, before sentence de-duplication: {np.mean([r['distinct_2_raw'] for r in FINAL_ROWS[:10]]):.3f}); "
                 f"the 1B free-form answers scored 0.419 in Assignment 1B. Short, cited answers naturally score higher, so this is a sanity check on the repetition problem, not a quality score."))

### D2 — Check the answers with an LLM judge
**Judge model:** it must differ from the generator. Generator = SmolLM2 (1B model) → judge = `Qwen/Qwen2.5-7B-Instruct`; generator = Qwen → judge = `meta-llama/Llama-3.1-8B-Instruct` (needs a Hugging Face token in Colab secret `HF_TOKEN`) with `microsoft/Phi-3.5-mini-instruct` as an un-gated fall-back. The judge is run **greedy (temperature 0)**, so the verdicts are repeatable.

**Judge prompt design:** it receives the question, the expected answer, the 3 chunks and the answer; asks **one question per check** (CORRECT vs the expected answer only; SUPPORTED vs the chunks only); defines every label exactly; says how to treat "Not found in the documents"; asks for a one-sentence reason per verdict and a quote of any unsupported statement; and fixes the output format so code can parse it.

In [ ]:
JUDGE_TEMPLATE = """You are a strict examiner marking one answer produced by an HR-policy question-answering system.
You get the QUESTION, the EXPECTED ANSWER, the 3 numbered CHUNKS the system was shown, and the system's ANSWER. Make two SEPARATE checks.

CHECK 1 - CORRECT. Compare the ANSWER with the EXPECTED ANSWER only (ignore the chunks, ignore your own knowledge).
  correct        = the answer states the same key fact(s) as the expected answer (different wording is fine, extra harmless detail is fine) and contradicts nothing in it.
  partly correct = the answer states at least one key fact of the expected answer but leaves out another key fact, or mixes it with a wrong detail.
  wrong          = the answer states none of the key facts of the expected answer, contradicts it, or is empty.
  If the answer is "Not found in the documents", CORRECT is "wrong", unless the expected answer itself says the information is not available; then it is "correct".

CHECK 2 - SUPPORTED. Use the CHUNKS only (ignore the expected answer, ignore your own knowledge).
  yes = every factual statement in the answer (every number, name, condition, duration, rule) can be found in, or follows directly from, the 3 chunks.
  no  = at least one factual statement is not in the chunks. Citation markers such as [2] are not statements.
  If the answer is "Not found in the documents", SUPPORTED is "yes" because it makes no claim.

Reply in EXACTLY this format, one item per line, with nothing before or after it:
CORRECT: <correct | partly correct | wrong>
CORRECT_REASON: <one sentence>
SUPPORTED: <yes | no>
SUPPORTED_REASON: <one sentence>
UNSUPPORTED_QUOTE: <exact quote of the unsupported statement, or NONE>

QUESTION: <<QUESTION>>
EXPECTED ANSWER: <<EXPECTED>>
CHUNKS:
<<CHUNKS>>
ANSWER: <<ANSWER>>"""
print("=== D2 JUDGE PROMPT ===\n"); print(JUDGE_TEMPLATE)

def judge_message(row):
    q = QI.get(row["id"], OOS)
    chunks = "\n".join(f"[{n}] {CHUNKS[i]['text']}" for n, i in enumerate(row["chunk_ids"], 1))
    exp = q["expected"] if row["id"] in QI else "The information is not in the documents; the correct reply is 'Not found in the documents'."
    return (JUDGE_TEMPLATE.replace("<<QUESTION>>", q["query"]).replace("<<EXPECTED>>", exp)
            .replace("<<CHUNKS>>", chunks).replace("<<ANSWER>>", row["answer"]))

def parse_judge(txt):
    def g(key):
        m = re.search(rf"^\s*{key}\s*:\s*(.+)$", txt, re.I | re.M); return m.group(1).strip() if m else None
    c, s = (g("CORRECT") or "").lower(), (g("SUPPORTED") or "").lower()
    c = "partly correct" if "partly" in c else "wrong" if "wrong" in c else "correct" if "correct" in c else None
    s = "yes" if s.startswith("yes") else "no" if s.startswith("no") else None
    return dict(correct=c, supported=s, correct_reason=g("CORRECT_REASON"), supported_reason=g("SUPPORTED_REASON"), quote=g("UNSUPPORTED_QUOTE"))

In [ ]:
safe_del("qmodel_g", "ft_model", "ft_base")
if FINAL_IS_QWEN:
    JUDGE_ID = "meta-llama/Llama-3.1-8B-Instruct"
    try:
        from huggingface_hub import login
        try:
            from google.colab import userdata; login(userdata.get("HF_TOKEN"))
        except Exception: pass
        jtok = AutoTokenizer.from_pretrained(JUDGE_ID)
    except Exception as e:
        print("Llama-3.1-8B-Instruct not available (", repr(e)[:90], ") -> Phi-3.5-mini-instruct"); JUDGE_ID = "microsoft/Phi-3.5-mini-instruct"; jtok = AutoTokenizer.from_pretrained(JUDGE_ID)
else:
    JUDGE_ID = "Qwen/Qwen2.5-7B-Instruct"; jtok = AutoTokenizer.from_pretrained(JUDGE_ID)
assert JUDGE_ID.split("/")[1].split("-")[0].lower() not in FINAL_NAME.lower(), "judge must differ from the generator"
jq4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
jmodel = AutoModelForCausalLM.from_pretrained(JUDGE_ID, quantization_config=jq4, torch_dtype=DTYPE, device_map={"": 0}).eval()
print("JUDGE MODEL:", JUDGE_ID, "| generator was:", FINAL_NAME)

JUDGE_OUT = {}
for row in FINAL_ROWS:
    msg = judge_message(row); res = None
    for attempt in range(2):
        txt, _ = chat(jmodel, jtok, msg + ("" if attempt == 0 else "\n\nIMPORTANT: reply ONLY with the five lines in the exact format above."), "template", 200)
        res = parse_judge(txt)
        if res["correct"] and res["supported"]: break
    res["raw"] = txt; JUDGE_OUT[row["id"]] = res
json.dump(JUDGE_OUT, open("D2_judge_raw.json", "w"), indent=1, ensure_ascii=False)
safe_del("jmodel")
print("judged", len(JUDGE_OUT), "answers; unparsed:", [k for k, v in JUDGE_OUT.items() if not (v["correct"] and v["supported"])])

**D2.1 — Judge results (one row per query) and summary.** Q11 (out-of-corpus) is judged too, but listed separately and not counted in the "out of 10".

In [ ]:
rows = []
for r in FINAL_ROWS[:10]:
    j = JUDGE_OUT[r["id"]]
    reason = f"{j['correct_reason']} {j['supported_reason']}" + (f" Unsupported: \"{j['quote']}\"" if j["quote"] and j["quote"].upper() != "NONE" else "")
    rows.append({"Query": r["id"], "Question": QI[r["id"]]["query"], "Correct": j["correct"], "Supported": j["supported"], "Reason": reason})
d2_df = pd.DataFrame(rows); pd.set_option("display.max_colwidth", 400); display(d2_df); d2_df.to_csv("D2_judge_results.csv", index=False)
s_df = pd.DataFrame([["Correct", int((d2_df.Correct == "correct").sum())], ["Partly correct", int((d2_df.Correct == "partly correct").sum())],
                     ["Wrong", int((d2_df.Correct == "wrong").sum())], ["Supported by the chunks", int((d2_df.Supported == "yes").sum())]],
                    columns=["Check", "Count out of 10"])
display(s_df); s_df.to_csv("D2_judge_summary.csv", index=False)
j11 = JUDGE_OUT["Q11"]; print(f"Q11 (not in corpus): answer = {FINAL_ROWS[-1]['answer']!r} -> judge: correct={j11['correct']}, supported={j11['supported']} | {j11['correct_reason']}")

**D2.2 — Spot-check the judge yourself.** The code picks 3 answers (one the judge calls correct+supported, one wrong/partly correct, one unsupported where available) and prints the question, expected answer, answer, the 3 chunks and the judge's verdict, plus a crude word-overlap hint. **Read them, then fill `MY_VERDICTS` in the next cell** with your own verdicts; the code counts agreements.

In [ ]:
def overlap_hint(row):
    cw = lambda t: {w for w in words(t) if len(w) > 3}
    a = cw(re.sub(r"\[\d+\]", "", row["answer"])); c = cw(" ".join(CHUNKS[i]["text"] for i in row["chunk_ids"]))
    return f"{100*len(a & c)/max(1,len(a)):.0f}% of the answer's content words occur in the 3 chunks"
ok = [r["id"] for r in FINAL_ROWS[:10] if JUDGE_OUT[r["id"]]["correct"] == "correct" and JUDGE_OUT[r["id"]]["supported"] == "yes"]
bad = [r["id"] for r in FINAL_ROWS[:10] if JUDGE_OUT[r["id"]]["correct"] in ("wrong", "partly correct")]
uns = [r["id"] for r in FINAL_ROWS[:10] if JUDGE_OUT[r["id"]]["supported"] == "no" and r["id"] not in bad]
SPOT = []
ALL = [r["id"] for r in FINAL_ROWS[:10]]
for pool in (ok, bad, uns, ok, bad, ALL, ALL, ALL):
    for x in pool:
        if x not in SPOT: SPOT.append(x); break
    if len(SPOT) == 3: break
for x in SPOT[:3]:
    r = next(r for r in FINAL_ROWS if r["id"] == x); j = JUDGE_OUT[x]
    print("=" * 100); print(f"{x}: {QI[x]['query']}\nEXPECTED: {QI[x]['expected']}\nANSWER:   {r['answer']}")
    for n, i in enumerate(r["chunk_ids"], 1): print(f"  [{n}] (chunk {i}, {'contains span' if is_hit(CHUNKS[i]['text'], x) else 'no span'}) {CHUNKS[i]['text']}")
    print(f"JUDGE -> correct: {j['correct']} | supported: {j['supported']} | {j['correct_reason']} {j['supported_reason']}\nHINT: {overlap_hint(r)}\n")
print("Spot-check queries:", SPOT[:3])

In [ ]:
# >>> FILL IN after reading the 3 printed answers and chunks: your own verdicts, e.g. {"Q1": ("correct", "yes"), ...}
# correct in {"correct","partly correct","wrong"}, supported in {"yes","no"}.  Use the 3 query ids printed above.
MY_VERDICTS = {}
if len(MY_VERDICTS) >= 3:
    agree = [(x, MY_VERDICTS[x][0] == JUDGE_OUT[x]["correct"], MY_VERDICTS[x][1] == JUDGE_OUT[x]["supported"]) for x in MY_VERDICTS]
    n_ok = sum(c and s for _, c, s in agree); n_chk = sum(c for _, c, _ in agree) + sum(s for _, _, s in agree)
    print(f"Agreement on both checks for {n_ok} of {len(agree)} answers ({n_chk} of {2*len(agree)} individual verdicts).")
    for x, c, s in agree:
        if not (c and s): print(f"DISAGREEMENT on {x}: mine={MY_VERDICTS[x]} vs judge=({JUDGE_OUT[x]['correct']}, {JUDGE_OUT[x]['supported']}) -> explain it in the markdown cell below")
else:
    print("!! Fill MY_VERDICTS for the 3 spot-check queries above, then re-run this cell.")

**D2.2 — My spot-check (write after running the cell above):**
*I read answers __, __ and __ with their chunks and agree with __ of the 3 judge verdicts. One disagreement (if any): the judge said … because …, but the chunk … shows …*

### D2.3 — Find the cause  |  D2.4 — Suggest a fix
The code picks the first answer the judge marked **wrong** (else partly correct, else unsupported, else the answer with the lowest overlap with its chunks), prints its 3 chunks, and decides *retrieval vs generation* from one test: **was the gold answer span inside the 3 chunks?** If yes the information was available and the generator failed; if no, retrieval (or chunking) failed. It also looks up where each search method ranked a chunk with the span.

In [ ]:
def failed_pick():
    for lab in ("wrong", "partly correct"):
        for r in FINAL_ROWS[:10]:
            if JUDGE_OUT[r["id"]]["correct"] == lab: return r
    for r in FINAL_ROWS[:10]:
        if JUDGE_OUT[r["id"]]["supported"] == "no": return r
    return min(FINAL_ROWS[:10], key=lambda r: float(overlap_hint(r).split("%")[0]))
F = failed_pick(); fid = F["id"]; fq = QI[fid]; fj = JUDGE_OUT[fid]; fi = [q["id"] for q in QUERIES].index(fid)
print(f"{fid}: {fq['query']}\nEXPECTED: {fq['expected']}\nANSWER:   {F['answer']}\nJUDGE: correct={fj['correct']} supported={fj['supported']} | {fj['correct_reason']} {fj['supported_reason']}\n")
print("Gold answer span:", fq["span"], "\n")
for n, i in enumerate(F["chunk_ids"], 1):
    print(f"--- chunk [{n}] id {i} | {CHUNKS[i]['source']} | tokens {CHUNKS[i]['n_tokens']} | {'CONTAINS the answer span' if is_hit(CHUNKS[i]['text'], fid) else 'does not contain the span'}\n{CHUNKS[i]['text']}\n")

gold_in = [is_hit(CHUNKS[i]["text"], fid) for i in F["chunk_ids"]]
span_chunks = [c["id"] for c in CHUNKS if is_hit(c["text"], fid)]
if any(gold_in):
    CAUSE = "GENERATION"
    CAUSE_TEXT = (f"**Cause: generation.** The needed sentence *was* in the chunks (chunk [{gold_in.index(True)+1}] contains the gold span), so retrieval and reranking did their job; the model still produced the answer above, "
                  f"which the judge marked correct={fj['correct']}, supported={fj['supported']}.")
    FIX_TEXT = ("**Fix:** change the prompt wording so the model must first copy the single sentence that answers the question (with its chunk number) and then answer only from that sentence, "
                "and keep the stronger instruct model as generator. This helps because the failure is in reading/using correct evidence, not in finding it; forcing an extractive step removes the room to add or alter details.")
elif not span_chunks:
    CAUSE = "CHUNKING"
    CAUSE_TEXT = "**Cause: retrieval (chunking).** No chunk of this chunk set contains the whole answer span, because fixed-size chunking cut the answer sentence at a boundary, so the needed information could never reach the generator."
    FIX_TEXT = "**Fix:** use sentence-aware chunking or 15-20% overlap so an answer sentence is never split. This helps because every sentence then appears whole in at least one chunk and can be retrieved."
elif any(is_hit(CHUNKS[i]["text"], fid) for i in BEST_FN(fq["query"], 10)):
    CAUSE = "RETRIEVAL (reranking step)"
    CAUSE_TEXT = ("**Cause: retrieval pipeline (reranking).** A chunk with the answer span was among the 10 candidates of the first-stage search, but the cross-encoder ranked it below the 3 chunks that were given to the generator, "
                  "so the needed information was not in the context.")
    FIX_TEXT = ("**Fix:** pass the top 5 (instead of the top 3) reranked chunks to the generator, or fuse the cross-encoder score with the first-stage score. This helps because the answer chunk is within the 10 candidates, "
                "so a slightly wider final context recovers it at a small cost in prompt tokens.")
else:
    CAUSE = "RETRIEVAL"
    pos = {m: RANKS[m][fi] for m in METHODS}
    CAUSE_TEXT = (f"**Cause: retrieval.** The chunk(s) with the answer exist ({span_chunks[:3]}) but none of the 3 chunks given to the generator contains the span, so the information was not in the context. "
                  f"First rank of an answer chunk: " + ", ".join(f"{m} {pos[m] if pos[m] else 'not in top 20'}" for m in METHODS) + ".")
    best_m = min((m for m in METHODS if pos[m]), key=lambda m: pos[m], default=None)
    FIX_TEXT = (f"**Fix:** retrieve with {best_m}, which ranks the answer chunk at #{pos[best_m]}, and/or pass the top 5 instead of the top 3 chunks to the generator. This helps because the evidence then reaches the prompt."
                if best_m else "**Fix:** use a larger chunk size or query rewriting, because no method found the answer chunk in its top 20.")
display(Markdown("**D2.3 —** " + CAUSE_TEXT)); display(Markdown("**D2.4 —** " + FIX_TEXT))
print("(Edit the wording if you disagree with the automatic diagnosis after reading the chunks above.)")

## Outputs and checklist

In [ ]:
final_rows = [["Embedding model", EMB_ID], ["Chunk size chosen (tokens)", CHUNK_SIZE], ["Total chunks", len(CHUNKS)], ["Best search method (Part B)", BEST_METHOD],
              ["Reranker", CE_ID], ["Rank change rate (%)", CHANGE], ["Hit@1 before / after rerank (%)", f"{H1_B_FROM_PART_B:.0f} / {H1_A:.0f}"],
              ["Generator used", FINAL_NAME], ["Judge model", JUDGE_ID],
              ["Judge: correct / partly / wrong", f"{(d2_df.Correct=='correct').sum()} / {(d2_df.Correct=='partly correct').sum()} / {(d2_df.Correct=='wrong').sum()}"],
              ["Judge: supported", int((d2_df.Supported == 'yes').sum())], ["Failure analysed", f"{fid} -> {CAUSE}"]]
pd.DataFrame(final_rows, columns=["Item", "Value"]).to_csv("Assignment2B_summary.csv", index=False); display(pd.DataFrame(final_rows, columns=["Item", "Value"]))
print("CSV files written to the working directory (download them with the notebook + HTML export).")